# Custom Inference Engine: Benchmark

Measures the engine (PagedAttention KV cache, radix prefix caching, continuous batching,
CUDA-graph decode) on a real Qwen2.5 checkpoint and saves **every number to a JSON file**.

**Setup (once):** on your PC run `python scripts/pack_for_colab.py` and upload
`dist/custom_inference_engine.zip` to the root of **My Drive**.

**Each run:** `Runtime -> Change runtime type -> GPU` (A100 for 7B, T4 is fine for 0.5B), then **Run all**.
Afterwards, **File -> Download .ipynb** (keeps the printed outputs) and keep the results JSON
from `MyDrive/inference_engine_results/`. Those two files are the evidence for any number you quote.

## 1. GPU check, engine code, dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
PROJECT_ZIP = f'{DRIVE}/custom_inference_engine.zip'   # from scripts/pack_for_colab.py
WORKDIR = '/content/custom_inference_engine'

shutil.rmtree(WORKDIR, ignore_errors=True)
with zipfile.ZipFile(PROJECT_ZIP) as zf:
    zf.extractall('/content')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Engine code extracted to', WORKDIR)

!pip install -q safetensors transformers huggingface_hub datasets matplotlib numpy tqdm

## 2. Configuration

Override any `.env` value here. `Qwen/Qwen2.5-7B-Instruct` on an A100, `Qwen/Qwen2.5-0.5B-Instruct` on a T4.

In [ ]:
MODEL_REPO = 'Qwen/Qwen2.5-7B-Instruct'
os.environ.update({
    'ENGINE_MODEL_REPO': MODEL_REPO,
    'ENGINE_MODEL_DIR': '/content/models/' + MODEL_REPO.split('/')[-1].lower(),
    'ENGINE_NUM_BLOCKS': '512',
    'ENGINE_BLOCK_SIZE': '16',
    'ENGINE_MAX_BATCH_SIZE': '16',
    'ENGINE_BENCHMARK_NUM_REQUESTS': '100',
    'ENGINE_BENCHMARK_MAX_NEW_TOKENS': '64',
})
RUN_LIVE_DASHBOARD = False   # optional ShareGPT traffic replay demo (section 4)
RUN_VLLM_BASELINE = False    # needs `pip install vllm` (large download)
PPL_MAX_WINDOWS = None       # None = full WikiText-2 test split (1024-token windows)

from config import reload_settings
SETTINGS = reload_settings(override=False)   # environment above wins over .env
print(SETTINGS)

## 3. Download model & build engine

In [ ]:
import json
import os
from huggingface_hub import snapshot_download
MODEL_REPO = SETTINGS.model_repo
MODEL_DIR = SETTINGS.model_dir


def _is_download_complete(model_dir: str) -> bool:
    if not os.path.isfile(os.path.join(model_dir, "config.json")):
        return False
    index_path = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.isfile(index_path):
        with open(index_path, "r", encoding="utf-8") as f:
            shards = set(json.load(f).get("weight_map", {}).values())
        return bool(shards) and all(os.path.isfile(os.path.join(model_dir, s)) for s in shards)
    return os.path.isfile(os.path.join(model_dir, "model.safetensors"))


if not _is_download_complete(MODEL_DIR):
    snapshot_download(
        MODEL_REPO,
        local_dir=MODEL_DIR,
        allow_patterns=[
            "*.safetensors", "model.safetensors.index.json",
            "config.json", "tokenizer.json",
            "tokenizer_config.json", "special_tokens_map.json",
            "vocab.json", "merges.txt",
        ],
    )
print(f"Model ({MODEL_REPO}) ready in {MODEL_DIR}: {sorted(os.listdir(MODEL_DIR))}")

In [ ]:
import json
import torch
from transformers import AutoTokenizer
from models.qwen2_model import Qwen2Model, Qwen2Config
from models.weight_loader import load_qwen2_weights
from core.engine import CustomInferenceEngine

with open(f"{MODEL_DIR}/config.json") as f:
    raw_config = json.load(f)

device = SETTINGS.resolve_device()
dtype = SETTINGS.resolve_torch_dtype()
config = Qwen2Config.from_hf_config(raw_config, dtype=dtype, max_model_len=SETTINGS.max_model_len)

device_label = torch.cuda.get_device_name(0) if device.startswith("cuda") and torch.cuda.is_available() else device.upper()
print(f"Model: {SETTINGS.model_name} | Device: {device} ({device_label}) | dtype: {dtype} | tie_word_embeddings: {config.tie_word_embeddings}")
if device.startswith("cuda") and torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_vram = getattr(props, "total_memory", getattr(props, "total_mem", 0))
    print(f"VRAM: {total_vram / 1e9:.1f} GB")

# Construct directly in target dtype to avoid 2x fp32 CPU RAM spike on large models
prev_dtype = torch.get_default_dtype()
torch.set_default_dtype(dtype)
try:
    model = Qwen2Model(config).to(device=device, dtype=dtype)
finally:
    torch.set_default_dtype(prev_dtype)

load_qwen2_weights(model, MODEL_DIR, dtype=dtype)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
stop_ids = set()
if tokenizer.eos_token_id is not None:
    stop_ids.add(tokenizer.eos_token_id)
for tok_str in ("<|im_end|>", "<|endoftext|>"):
    tid = tokenizer.convert_tokens_to_ids(tok_str)
    if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id:
        stop_ids.add(tid)

engine = CustomInferenceEngine(
    model=model,
    num_blocks=SETTINGS.num_blocks,
    block_size=SETTINGS.block_size,
    max_batch_size=SETTINGS.max_batch_size,
    max_batched_tokens=SETTINGS.max_batched_tokens,
    max_model_len=SETTINGS.max_model_len,
    stop_token_ids=stop_ids,
    enable_prefix_caching=SETTINGS.enable_prefix_caching,
    use_cuda_graph=SETTINGS.use_cuda_graph and device.startswith("cuda"),
    temperature=SETTINGS.temperature,
)
print(f"Model loaded: {model.num_parameters / 1e9:.2f}B params | KV cache: {engine.get_stats()['kv_cache_mb']:.1f} MB")

In [ ]:
# Quick sanity check: greedy completion
prompt_ids = tokenizer.encode("The capital of France is")
engine.clear_prefix_cache()
out = engine.generate([prompt_ids], max_tokens=24)[0]
print(tokenizer.decode(prompt_ids + out.output_tokens, skip_special_tokens=True))
print(f"TTFT={out.ttft_ms:.2f} ms  TPOT={out.tpot_ms:.2f} ms")

## 4. (Optional) Live ShareGPT traffic dashboard

Streams real user prompts from the **ShareGPT** dataset (with Poisson request arrivals across `ShareGPT/Coding`, `ShareGPT/Science`, `ShareGPT/SysDesign`, and `ShareGPT/Math&ML`) into `CustomInferenceEngine` and renders an in-place, live-updating terminal dashboard showing:
- **Incoming requests & sources** (which ShareGPT domain and user ID each request comes from, arrival timestamp, prompt token count)
- **Continuous batching queues** (Waiting vs. Prefill vs. Decode sequences per iteration)
- **Paged KV Cache block occupancy bar** and **Radix Prefix Cache hit rate**
- **Live token-by-token streaming output** for every active request on the A100 GPU

In [ ]:
if RUN_LIVE_DASHBOARD:
    from telemetry.live_monitor import load_sharegpt_trace, LiveTrafficMonitor

    sharegpt_trace = load_sharegpt_trace(
        num_requests=SETTINGS.trace_num_requests,
        qps=SETTINGS.trace_qps,
        max_new_tokens=SETTINGS.trace_max_new_tokens,
        download_from_hf=True,
        stream_bytes=SETTINGS.trace_stream_bytes,
    )
    n_unique = len({r.full_prompt for r in sharegpt_trace})
    print(f"Trace: {len(sharegpt_trace)} requests | {n_unique} unique prompts "
          f"| replay x{len(sharegpt_trace)/max(n_unique,1):.1f}")

    monitor = LiveTrafficMonitor(
        engine=engine,
        tokenizer=tokenizer,
        refresh_every_n_steps=1,
        step_pause_s=0.01,   # 10 ms UI pacing so you can actually watch the stream
    )
    live_completed = monitor.run(sharegpt_trace, burst_initial=SETTINGS.trace_burst_initial)

    stats = engine.get_stats()
    avg_decode = monitor._decode_batch_sum / max(monitor._decode_batch_samples, 1)
    print("\n--- Live telemetry summary ---")
    print(f"Peak waiting/running : {monitor.peak_waiting}/{monitor.peak_running}")
    print(f"Peak prefill/decode  : {monitor.peak_prefill}/{monitor.peak_decode}  (avg decode {avg_decode:.2f})")
    print(f"KV blocks peak       : {monitor.peak_used_blocks}/{engine.block_manager.num_blocks} "
          f"({monitor.peak_used_blocks/engine.block_manager.num_blocks*100:.1f}%)")
    print(f"Cached prefix blocks : {engine.block_manager.prefix_cache.num_cached_blocks} | "
          f"evictions: {stats['evicted_blocks']} | hit rate: {stats['prefix_cache_hit_rate']*100:.1f}%")

## 5. Benchmark

Single-stream latency, prefix-cache validation, continuous-batching throughput with KV-memory
accounting, prefill throughput, CUDA-graph ablation, capacity, decode fidelity, then WikiText-2 perplexity.

In [ ]:
from benchmarks.workload import SHARED_SYSTEM_PROMPT, build_benchmark_prompts
from benchmarks.harness import BenchmarkHarness, perplexity, wikitext2_token_ids

PROMPTS = build_benchmark_prompts(SETTINGS.benchmark_num_requests)
harness = BenchmarkHarness(engine, tokenizer, SETTINGS)
results = harness.run(PROMPTS, max_new_tokens=SETTINGS.benchmark_max_new_tokens,
                      shared_prefix=SHARED_SYSTEM_PROMPT)

WIKI_IDS = wikitext2_token_ids(tokenizer)
results.perplexity = perplexity(harness.custom_logits_fn(), WIKI_IDS, engine.device,
                                window=1024, max_windows=PPL_MAX_WINDOWS)
print(f"WikiText-2 perplexity (custom engine): {results.perplexity['ppl']:.3f}")

## 6. Baselines on the same GPU (HF Transformers, optional vLLM)

In [ ]:
import gc
from benchmarks.baselines import run_hf_baseline, run_vllm_baseline

# Free the custom engine so each baseline gets the whole GPU
engine.release_memory()
for name in ("harness", "monitor", "engine", "model"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

hf = run_hf_baseline(MODEL_DIR, tokenizer, PROMPTS, SETTINGS.benchmark_max_new_tokens,
                     SETTINGS.max_batch_size, dtype, device, SETTINGS.max_model_len,
                     ppl_token_ids=WIKI_IDS, ppl_max_windows=PPL_MAX_WINDOWS)
results.perplexity["hf_ppl"] = hf.pop("ppl")
results.baselines["hf_transformers"] = hf

if RUN_VLLM_BASELINE:
    v = run_vllm_baseline(MODEL_DIR, PROMPTS, SETTINGS.benchmark_max_new_tokens,
                          SETTINGS.max_batch_size, dtype, SETTINGS.max_model_len)
    if v:
        results.baselines["vllm"] = v

## 7. Results (saved to Drive)

In [ ]:
import datetime
from benchmarks.report import print_results, print_claims, plot_results, save_results_json

print_results(results)
print_claims(results)

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
gpu_tag = (results.env.get('gpu') or 'cpu').replace(' ', '_')
out_dir = f'{DRIVE}/inference_engine_results'
save_results_json(results, f'{out_dir}/{SETTINGS.model_name}_{gpu_tag}_{stamp}.json')
plot_results(results, f'{out_dir}/{SETTINGS.model_name}_{gpu_tag}_{stamp}.png')